<p align="center"><img src="logo.png" width="160"/></p>

# IonScell — Single-Cell Mass Spectrometry Imaging

This notebook runs the full IonScell pipeline by importing the **** class from .
The class is **not** defined here — edit  to modify the analysis logic.

---
**Pipeline steps:**
0. Environment & imports
1. Load imzML data
2. Build data cube (m/z binning + normalization)
3. Visualise TIC
4. Adaptive TIC thresholding
5. Cell segmentation
6. Inspect & clean segmentation
7. Extract cell spectra
8. Quality control
9. Soft clustering (GMM / FCM)
10. Clone quality assessment & visualisation
11. Differential analysis
12. Lipid annotation & export


## Step 0 — Environment & Imports
Install missing packages when running in Google Colab, then import the pipeline class.


In [ ]:
# ── Colab auto-install (skipped on local Jupyter) ───────────────────────────
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import subprocess
    packages = ["pyimzml", "umap-learn", "scikit-image", "plotly", "tqdm", "scikit-fuzzy"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + packages)
    # Download pipeline class from GitHub
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/your-org/ionscell/main/ionscell_pipeline.py",
        "ionscell_pipeline.py"
    )
    print("✅  IonScell dependencies installed")

# ── Imports ──────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, ".")

from ionscell_pipeline import SCMSIPipeline, LipidAnnotator
import numpy as np
import pandas as pd
print("✅  IonScell ready")


---
## Step 1 — Load imzML Data
Point  to your  file (the  binary must be in the same folder).


In [ ]:
# ── Google Colab: upload your files ─────────────────────────────────────────
if IN_COLAB:
    from google.colab import files
    print("Select both .imzML and .ibd files:")
    uploaded = files.upload()
    imzml_path = [k for k in uploaded if k.endswith(".imzML")][0]
else:
    imzml_path = "data/sample.imzML"   # ← change to your file path

# ── Load ─────────────────────────────────────────────────────────────────────
pipe = SCMSIPipeline(verbose=True)
pipe.load_imzml(imzml_path)


---
## Step 2 — Build Data Cube
Choose **fixed** binning (simple, fast) or **adaptive** binning (matches instrument resolution).
Normalization options:  (recommended), , , or .


In [ ]:
# ── Option A: Fixed binning (recommended for clustering) ─────────────────────
pipe.build_datacube(
    mz_min=350,
    mz_max=1100,
    mz_bin=0.1,
    normalization="TIC",   # TIC | RMS | MEDIAN | None
)

# ── Option B: Adaptive binning (recommended for annotation) ──────────────────
# pipe.build_adaptive_mz_axis(
#     mz_min=350, mz_max=1100,
#     target_resolving_power=20000,  # or ppm_bin=50
# )
# pipe.build_datacube(normalization="TIC")

print(f"Data cube shape: {pipe.data_cube.shape}")


---
## Step 3 — Visualise TIC
Inspect the Total Ion Current image. Bright regions = high spectral intensity = cells.


In [ ]:
pipe.show_TIC_plotly(
    sigma=0.5,        # Gaussian smoothing (0 = none)
    cmap="magma",     # colormap: magma | viridis | jet | gray
    zoom_factor=1.0,
)


---
## Step 4 — Adaptive TIC Thresholding
IonScell tests 8 threshold algorithms and selects the best one automatically.
The diagnostic histogram shows all candidates — the solid line is the selected threshold.


In [ ]:
threshold, diagnostics = pipe.auto_TIC_threshold(
    method="auto",          # auto | gmm | otsu | triangle | li | yen | percentile
    show_diagnostics=True,
)

print(f"
Selected threshold: {threshold:.1f}")

# To override manually:
# threshold = 500.0


---
## Step 5 — Cell Segmentation
Watershed (recommended for separated cells) or DFS (better for dense/touching cells).


In [ ]:
labels = pipe.watershed_or_dfs_from_mask(
    TIC_threshold=threshold,
    method="watershed",     # watershed | dfs
    sigma=0.5,
    footprint=(5, 5),       # local maxima neighbourhood
    min_cell_size=4,        # minimum cell size (pixels)
    max_cell_size=None,     # None = no upper limit
    pixel_size_um=10,
    size_unit="px",         # px | um
    remove_isolated=True,
    show_plotly=True,
)
print(f"Cells detected: {int(labels.max())}")


---
## Step 6 — Inspect & Clean Segmentation
Hover over contours to see cell IDs. Remove artefacts (dust, crystals, vessel lumens) by ID.


In [ ]:
# List cell IDs to remove (empty list = nothing to remove)
artefact_ids = []   # e.g. [3, 17, 42]

if artefact_ids:
    pipe.remove_cells(remove_ids=artefact_ids, show_plotly=True)
else:
    print("No cells to remove.")


---
## Step 7 — Extract Cell Spectra
Aggregate all pixels belonging to each cell into one representative spectrum.


In [ ]:
df_cells = pipe.extract_cell_spectra(agg="mean")  # mean | median | sum
print(f"{len(df_cells)} cells extracted, {df_cells.shape[1]} columns")
df_cells[["cell_id","area_px","area_um","centroid_row","centroid_col"]].head()


---
## Step 8 — Quality Control
Compute per-cell QC metrics (SNR, sparsity, entropy, …), visualise the distributions,
and optionally remove low-quality cells.


In [ ]:
# Compute metrics (adds qc_* columns to df_cells)
qc = pipe.compute_spectral_quality_metrics()

# Visualise and flag
flagged = pipe.filter_low_quality_cells(
    snr_threshold=2.0,
    intensity_percentile=10,
    sparsity_min=0.05,
    auto_remove=False,   # set True to remove flagged cells automatically
    show_plot=True,
)
print(f"{len(flagged)} cells flagged as low quality")


---
## Step 9 — Soft Clustering
**GMM** assigns each cell a probability vector across clusters (soft membership).
Set  for automatic selection via BIC and silhouette score.


In [ ]:
# ── GMM (recommended) ────────────────────────────────────────────────────────
memberships, labels, confidence = pipe.compute_soft_clustering(
    n_clusters=None,        # None = automatic
    max_clusters=8,
    covariance_type="tied",
    use_umap_space=True,
    n_neighbors=15,
    random_state=42,
)

# ── FCM alternative ──────────────────────────────────────────────────────────
# memberships, labels, confidence = pipe.compute_fuzzy_cmeans(
#     max_clusters=8, m=2.0, use_umap_space=True
# )

n_clones = int(pipe.last_cell_spectra["Class"].nunique())
mean_conf = float(confidence.mean())
print(f"Clusters found: {n_clones}  |  Mean confidence: {mean_conf:.3f}")


---
## Step 10 — Clone Quality Assessment & Visualisation


In [ ]:
# Quality report
quality = pipe.assess_clone_quality(min_confidence=0.6, min_cells_per_clone=5)

# UMAP — clone colours, marker opacity = confidence
pipe.show_umap_with_quality_overlay()

# Spatial overlay on TIC
pipe.overlay_clusters_on_image_plotly(opacity_by_confidence=True)

# Cell contours on white background
pipe.plot_cell_contours_by_clone()


In [ ]:
# Mean spectra ± SD per clone
pipe.plot_clone_spectra_with_uncertainty(mode="overlay", normalize=True)

# Single-ion spatial map — change mz_value to your ion of interest
pipe.show_mz_distribution_in_cells_plotly(mz_value=760.59, tolerance=0.1, smoothing_sigma=1.0)

# Multi-ion overlay (up to 6 ions)
pipe.plot_multi_ion_overlay(
    mz_list=[760.59, 782.57, 810.60],
    colors=["#ff4d6d", "#00f5d4", "#fee440"],
)

# Violin distribution by clone
pipe.plot_distribution_by_clone(mz_value=760.59, plot_type="violin")


In [ ]:
# ── Clone management (run only if needed) ────────────────────────────────────
# Merge two similar clones
# pipe.combine_clones([1, 3], new_label=1)

# Remove a spurious clone
# pipe.remove_clone(clone_id=2)


---
## Step 11 — Differential Analysis
Kruskal-Wallis test (non-parametric) across all clones, with Benjamini-Hochberg FDR correction.
Results include a volcano plot (log₂FC vs −log₁₀ adj-p) and a Z-score heatmap.


In [ ]:
top_markers, full_results = pipe.run_differential_analysis(
    top_n=30,
    pval_threshold=0.05,
    fc_threshold=1.5,
    method="kruskal",     # kruskal | anova
    show_volcano=True,
    show_heatmap=True,
)

print(f"{len(top_markers)} significant marker ions")
top_markers.head(10)


---
## Step 12 — Lipid Annotation & Export
Annotate differential ions against the embedded LIPID MAPS / HMDB database.
No internet connection required.


In [ ]:
# ── Annotation ───────────────────────────────────────────────────────────────
ann_df, enrichment = pipe.annotate_ions(
    mode="neg",               # pos | neg | both
    ppm_tolerance=10.0,
    top_markers_df=top_markers,
    show_table=True,
    show_enrichment=True,
)

# Free-text database search
# LipidAnnotator(mode="both").search("ceramide")


In [ ]:
# ── Export ───────────────────────────────────────────────────────────────────
os.makedirs("results", exist_ok=True)

# CSV (metadata + spectra + cluster assignments)
pipe.export_cells_to_csv(
    "results/cells.csv",
    include_spectra=True,
    round_intensities=4,
)

# imzML (segmented cell pixels)
pipe.export_cells_to_imzML("results/cells_segmented.imzML")

# Download in Colab
if IN_COLAB:
    from google.colab import files
    files.download("results/cells.csv")
